# Track B - Classical machine learning

This notebook compares TF-IDF + Naive Bayes, logistic regression, linear SVM, and random forest. Model selection uses repeated duplicate-safe grouped folds from the official training split. The official test split is evaluated once, after selection. The command-line workflow in `examples/classical_ml.py` additionally writes bootstrap, error-analysis, and temporal-robustness artifacts.

In [ ]:
import json
from pathlib import Path

import pandas as pd

from ai4se.classical import (
    AVAILABLE_MODELS,
    TfidfConfig,
    make_classical_estimator_factory,
    model_display_name,
)
from ai4se.evaluation import (
    evaluate_cross_validation,
    evaluate_holdout_by_repository,
    save_evaluation,
)
from ai4se.reporting import result_rows, write_result_tables
from ai4se.service import IssueDataService
from ai4se.statistical_analysis import (
    result_repository_score,
    summarise_repeated_results,
)

## Configuration

Word and character TF-IDF are combined. A training-only 36-configuration ablation selected raw normalized text, 400-word truncation, and title weight 3. Five CV seeds measure the stability of the model-family decision.

In [ ]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RANDOM_STATE = 42
SEEDS = [42, 43, 44, 45, 46]
N_SPLITS = 5
CLEANING_LEVEL = "raw"
MAX_WORDS = 400
TITLE_WEIGHT = 3
OUTPUT_DIRECTORY = PROJECT_ROOT / "results" / "classical"

tfidf = TfidfConfig()
service = IssueDataService.from_loader(kind="memory")
train = service.prepare(
    "train",
    level=CLEANING_LEVEL,
    max_words=MAX_WORDS,
    title_weight=TITLE_WEIGHT,
)
len(train), tfidf

## Duplicate-safe model comparison

The TF-IDF vocabulary and IDF weights are learned inside each fold because vectorisation is part of each model pipeline.

In [ ]:
cv_results = {}
repeated_summaries = {}
for model in AVAILABLE_MODELS:
    runs = [
        evaluate_cross_validation(
            train,
            make_classical_estimator_factory(
                model, tfidf=tfidf, random_state=seed
            ),
            model_name=model_display_name(model),
            n_splits=N_SPLITS,
            random_state=seed,
            metadata={
                "track": "B",
                "cleaning_level": CLEANING_LEVEL,
                "max_words": MAX_WORDS,
                "title_weight": TITLE_WEIGHT,
                "tfidf": tfidf.to_dict(),
                "cross_validation_seeds": SEEDS,
            },
        )
        for seed in SEEDS
    ]
    cv_results[model] = runs[0]
    repeated_summaries[model] = summarise_repeated_results(runs)
    save_evaluation(runs[0], OUTPUT_DIRECTORY / f"{model}-cross-validation.json")
    summary_path = OUTPUT_DIRECTORY / f"{model}-repeated-cross-validation.json"
    summary_path.write_text(
        json.dumps(repeated_summaries[model], indent=2) + "\n",
        encoding="utf-8",
    )

In [ ]:
ranking = pd.DataFrame(
    [
        {
            "model": model,
            "mean_cross_repository_weighted_f1": summary["mean"],
            "standard_deviation": summary["standard_deviation"],
        }
        for model, summary in repeated_summaries.items()
    ]
).sort_values("mean_cross_repository_weighted_f1", ascending=False)
ranking

## Final official evaluation

Only the cross-validation winner is now trained on each repository's complete training subset and evaluated on that repository's official test subset.

In [ ]:
selected_model = ranking.iloc[0]["model"]
test = service.prepare(
    "test",
    level=CLEANING_LEVEL,
    max_words=MAX_WORDS,
    title_weight=TITLE_WEIGHT,
)
official_result = evaluate_holdout_by_repository(
    train,
    test,
    make_classical_estimator_factory(
        selected_model, tfidf=tfidf, random_state=RANDOM_STATE
    ),
    model_name=model_display_name(selected_model),
    random_state=RANDOM_STATE,
    metadata={"track": "B", "selected_by_cross_validation": True},
)
save_evaluation(
    official_result,
    OUTPUT_DIRECTORY / f"{selected_model}-official-holdout.json",
)
result_repository_score(official_result)

In [ ]:
all_results = [*cv_results.values(), official_result]
rows = [row for result in all_results for row in result_rows(result)]
write_result_tables(
    rows, PROJECT_ROOT / "results" / "tables", stem="classical_ml"
)